# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/busrayildirim0/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
import pandas as pd
import numpy as np

# Veri seti yolu
DATA_PATH = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with shape: {df.shape}")

# Hedef proxy: İçeriğin düşüş trendinde olup olmadığı
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

Loaded dataset with shape: (30000, 44)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Sinyal 1: CTR Beklenti Açığı (Position 1-10 arası olup CTR < %2 olanlar)
df["ctr_underperforming"] = (
    (df["avg_position"] <= 10.0) & (df["ctr"] < 0.02)
).astype(int)

bucket_signal_1 = df.groupby("ctr_underperforming").agg(
    n=("is_declining", "count"),
    declining_count=("is_declining", "sum"),
    decline_rate=("is_declining", "mean")
).reset_index()

print("--- Signal 1: CTR Underperformance on Page 1 ---")
display(bucket_signal_1)

# Sinyal 2: Position Tier Segment Dağılımı
bucket_signal_2 = df.groupby("position_tier").agg(
    n=("is_declining", "count"),
    declining_count=("is_declining", "sum"),
    decline_rate=("is_declining", "mean")
).reset_index().sort_values(by="decline_rate", ascending=False)

print("\n--- Signal 2: SERP Position Tier ---")
display(bucket_signal_2)

--- Signal 1: CTR Underperformance on Page 1 ---


,ctr_underperforming,n,declining_count,decline_rate
0,0,24147,13665,0.565909
1,1,5853,2597,0.443704



--- Signal 2: SERP Position Tier ---


,position_tier,n,declining_count,decline_rate
3,striking,7304,4452,0.609529
1,page_1,11814,6730,0.569663
2,page_3_5,7242,4067,0.561585
0,deep,1319,454,0.344200
4,top_3,2321,559,0.240844


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# Klasörü hazırla
os.makedirs("work/outputs", exist_ok=True)

# 1. Kural Fonksiyonu
def evaluate_baseline_action(row):
    score = 0.0
    reason_code = "NO_ACTION"
    action_label = "MONITOR"

    # Kural 1: İlk sayfada ama CTR düşük -> Metadata / Title Optimizasyonu
    if row["avg_position"] <= 10.0 and row["ctr"] < 0.02:
        score = 85.0 + min(15.0, (row["search_volume"] / 1000.0) * 2)
        reason_code = "HIGH_RANK_LOW_CTR"
        action_label = "REFRESH_METADATA"

    # Kural 2: Striking distance (Pozisyon 11-20) + Yüksek Rekabet -> İçerik Genişletme
    elif 10.0 < row["avg_position"] <= 20.0 and row["competition_level"] == "HIGH":
        score = 70.0 + min(15.0, (row["word_count"] < 1000) * 10)
        reason_code = "STRIKING_DIST_HIGH_COMP"
        action_label = "EXPAND_CONTENT"

    # Kural 3: Kötü Pozisyon (>20) + Düşük Scroll/Etkileşim -> Kapsamlı Revizyon
    elif row["avg_position"] > 20.0 and row["scroll_rate"] < 0.20:
        score = 50.0 + min(20.0, (1.0 - row["scroll_rate"]) * 20)
        reason_code = "LOW_POSITION_LOW_ENGAGEMENT"
        action_label = "FULL_REWRITE"

    else:
        score = 10.0
        reason_code = "STABLE_PERFORMANCE"
        action_label = "MONITOR"

    return pd.Series([round(score, 2), reason_code, action_label])

# 2. Kuralları Uygula
df[["baseline_score", "reason_code", "action_label"]] = df.apply(evaluate_baseline_action, axis=1)

# 3. Sıralı Kuyruk (Ranked Queue) Oluşturma
queue_cols = [
    "content_id", "client_id", "baseline_score", "action_label",
    "reason_code", "search_volume", "avg_position", "ctr", "word_count"
]
ranked_queue = df[queue_cols].sort_values(by="baseline_score", ascending=False).reset_index(drop=True)

# 4. work/outputs/baseline_action_score.csv olarak kaydet
OUTPUT_CSV = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(OUTPUT_CSV, index=False)
print(f"Ranked queue written to {OUTPUT_CSV} ({len(ranked_queue)} rows)")

Ranked queue written to work/outputs/baseline_action_score.csv (30000 rows)


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
top_10 = ranked_queue.head(10)
display(top_10)

,content_id,client_id,baseline_score,action_label,reason_code,search_volume,avg_position,ctr,word_count
0,content_1d4f4025c930,client_25fc0e7096,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,3510.0
1,content_5a3e876cf7f7,client_d4735e3a26,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,776.0
2,content_4be930227848,client_d4735e3a26,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,3.5,0.0,730.0
3,content_92a5d2709aa9,client_25fc0e7096,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,4677.0
4,content_d10c7372129c,client_624b60c58c,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,7.4,0.0,3542.0
5,content_a3af3b8346d8,client_25fc0e7096,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,4384.0
6,content_ba7082c9436c,client_25fc0e7096,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,1383.0
7,content_3e4fb8745f38,client_25fc0e7096,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,1454.0
8,content_41ffdf98a18e,client_624b60c58c,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,9.5,0.0,4366.0
9,content_40fa7c504e5a,client_25fc0e7096,100.0,REFRESH_METADATA,HIGH_RANK_LOW_CTR,NaN,0.0,0.0,4072.0


In [5]:
import json

metrics_receipt = {
    "task": "ML-07",
    "total_rows_scored": int(len(df)),
    "actions_distribution": df["action_label"].value_counts().to_dict(),
    "top_reason_codes": df["reason_code"].value_counts().to_dict(),
    "avg_baseline_score": float(round(df["baseline_score"].mean(), 2)),
    "signals_checked": ["ctr_underperformance", "position_tier"]
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics_receipt, f, indent=2)

print("Saved metrics to work/outputs/baseline_metrics.json")

Saved metrics to work/outputs/baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.